# Cutout ingestion workflow

This notebook walks through **ingesting galaxy cutouts** from FITS (one stamp per file in the demo) into HEALPix-partitioned **Zarr v3** under `cutouts/<survey>/`, validating the survey, and reading back with `CutoutAccessor`.

**Modes**

1. **Self-contained demo** — builds two tiny single-HDU FITS cutouts in a temporary lake.
2. **Your data** — set `USE_TEMP_LAKE = False` and point `LAKE_ROOT` plus a list of FITS paths.

Requirements for each FITS:

- **Image** in a data HDU (here: Primary HDU) with shape `(H, W)` or `(B, H, W)` if you set `band_axis`.
- **Header** keywords for sky position (`RA`, `DEC` by default) and integer **`SOURCE_ID`** or **`OBJ_ID`** (see `_extract_records_from_hdul` in `fits_to_zarr.py`).
- A minimal **WCS** on the header (`CRVAL*`, `CRPIX*`, `CD*_*`, `NAXIS*`) for round-trip to FITS.

After cutouts exist, you normally **`update_index_column`** on the **Parquet catalog** for the same survey so `_cutout_index` points at the Zarr row (optional cell below).

In [ ]:
from __future__ import annotations

import shutil
import tempfile
from pathlib import Path

import numpy as np
from astropy.io import fits

from data_lake.ingest.fits_to_zarr import ingest_cutouts_from_fits
from data_lake.ingest.validate_cutout_ingest import run_validation
from data_lake.io.cutouts import CutoutAccessor

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

USE_TEMP_LAKE = True
SURVEY = "demo_cutouts_nb"
NORDER = 5
RA_COL = "RA"
DEC_COL = "DEC"

if USE_TEMP_LAKE:
    LAKE_ROOT = Path(tempfile.mkdtemp(prefix="data_lake_cutout_nb_"))
    CUTOUT_DIR = LAKE_ROOT / "input_cutouts"
else:
    LAKE_ROOT = Path("/data/lake")
    CUTOUT_DIR = Path("/data/raw/cutouts_fits")  # directory of per-source FITS files

print("LAKE_ROOT:", LAKE_ROOT.resolve())

## 1. (Demo) Write minimal FITS cutouts

Each file is a **PrimaryHDU** 2-D image with `RA`, `DEC`, `OBJ_ID`, and a small TAN WCS (same pattern as `tests/test_cutout_ingest.py`).

In [ ]:
def write_demo_cutout_fits(path: Path, *, sid: int, ra: float, dec: float, seed: int) -> None:
    rng = np.random.RandomState(seed)
    data = rng.rand(16, 16).astype(np.float32)
    hdu = fits.PrimaryHDU(data)
    hdr = hdu.header
    hdr[RA_COL] = ra
    hdr[DEC_COL] = dec
    hdr["OBJ_ID"] = sid
    hdr["CTYPE1"] = "RA---TAN"
    hdr["CTYPE2"] = "DEC--TAN"
    hdr["CRVAL1"] = ra
    hdr["CRVAL2"] = dec
    hdr["CRPIX1"] = 8.0
    hdr["CRPIX2"] = 8.0
    hdr["CD1_1"] = -1.0 / 3600.0
    hdr["CD1_2"] = 0.0
    hdr["CD2_1"] = 0.0
    hdr["CD2_2"] = 1.0 / 3600.0
    path.parent.mkdir(parents=True, exist_ok=True)
    hdu.writeto(path, overwrite=True)


if USE_TEMP_LAKE:
    CUTOUT_DIR.mkdir(parents=True, exist_ok=True)
    f1 = CUTOUT_DIR / "src_5001.fits"
    f2 = CUTOUT_DIR / "src_5002.fits"
    # Nearby sky positions → usually same HEALPix tile at Norder=5
    write_demo_cutout_fits(f1, sid=5001, ra=200.0, dec=0.0, seed=1)
    write_demo_cutout_fits(f2, sid=5002, ra=200.02, dec=0.01, seed=2)
    DEMO_FILES = [f1, f2]
    print("Wrote:", *DEMO_FILES, sep="\n  ")
else:
    DEMO_FILES = sorted(CUTOUT_DIR.glob("*.fits"))
    assert DEMO_FILES, f"No FITS under {CUTOUT_DIR}"
    print(len(DEMO_FILES), "FITS files to ingest")

## 2. Ingest FITS → Zarr (`ingest_cutouts_from_fits`)

Returns **`source_id → local row index`** in the HEALPix tile for this ingest call. Use `on_duplicate_source_id`:

- **`append`** — re-runs can append duplicate `source_id` rows (avoid for production unless you know why).
- **`error`** — refuse if any `source_id` already exists in that tile’s Zarr.
- **`skip`** — only append sources not yet present.

In [ ]:
index_maps: list[dict[int, int]] = []
for fp in DEMO_FILES:
    m = ingest_cutouts_from_fits(
        source_path=fp,
        output_root=LAKE_ROOT,
        survey_name=SURVEY,
        ra_col=RA_COL,
        dec_col=DEC_COL,
        norder=NORDER,
        band_names=["band0"],  # single-band stamp; stored as Zarr attrs
        on_duplicate_source_id="append",
    )
    index_maps.append(m)
    print(fp.name, "→", m)

merged = {}
for m in index_maps:
    merged.update(m)
print("merged index map:", merged)

## 3. Validate the cutout survey

Checks `cutout_info.json`, optional checkpoint sidecars, and a sample of Zarr tiles (`images` / `source_id` / `wcs` shapes).

In [ ]:
rep = run_validation(LAKE_ROOT, SURVEY, max_tiles=20)
print("errors:", rep.errors)
print("warnings:", rep.warnings)
assert rep.ok(strict=False), "Validation failed — inspect errors above."

## 4. Read back with `CutoutAccessor`

Without a catalog, the accessor **scans Zarr tiles** on disk to resolve `source_id` (fine for demos). For production, open a matching **`CatalogAccessor`** and pass `catalog_accessor=` so lookup uses `_cutout_index` and HEALPix columns.

In [ ]:
sid = next(iter(merged))
acc = CutoutAccessor(LAKE_ROOT, SURVEY, norder=NORDER)
img, wcs = acc.get_cutout(sid)
print("source_id", sid, "image shape", img.shape, "dtype", img.dtype)
print("WCS:", wcs)
hdr = wcs.to_fits_header()
print(hdr["CRVAL1"], hdr["CRVAL2"], hdr["NAXIS1"], hdr["NAXIS2"])

## 5. (Optional) Patch `_cutout_index` on the Parquet catalog

Run this only if you already have **`catalogs/<same_survey_name>/`** from `ingest_catalog`. It writes the Zarr row index into each Parquet tile for fast lookup.

In [ ]:
cat_root = LAKE_ROOT / "catalogs" / SURVEY
if cat_root.is_dir():
    from data_lake.ingest.update_catalog_indices import update_index_column

    n_tiles = update_index_column(
        lake_root=LAKE_ROOT,
        survey_name=SURVEY,
        source_id_to_index=merged,
        kind="cutout",
        norder=NORDER,
    )
    print("Updated Parquet tile files:", n_tiles)
else:
    print("No catalog at", cat_root, "— skip index patch (ingest a catalog first).")

## 6. CLI equivalents

With `DATA_LAKE_CONFIG` set (or pass the lake root explicitly):

```bash
export DATA_LAKE_CONFIG=/path/to/my_lake/lake_config.toml

dl-ingest-cutouts /data/raw/stamp_001.fits --survey my_cutouts \\
  --ra-col RA --dec-col DEC \\
  --band-names r,i,z \\
  --on-duplicate skip

dl-validate-cutout-ingest --survey my_cutouts
```

Many files + checkpoint:

```bash
find /data/stamps -name '*.fits' > cutout_files.txt
dl-ingest-cutouts-from-list cutout_files.txt --survey my_cutouts \\
  --ra-col RA --dec-col DEC --on-duplicate skip
```

## 7. (Optional) Resolve `LAKE_ROOT` from config

```python
from data_lake.config import LakeConfig
LAKE_ROOT = LakeConfig.discover().lake.root
```

## 8. (Demo only) Remove temporary lake

In [ ]:
if USE_TEMP_LAKE:
    shutil.rmtree(LAKE_ROOT, ignore_errors=True)
    print("Removed temp lake:", LAKE_ROOT)